# Técnicas avanzadas de prompting para análisis literario

**Autor:** Jheison Martinez Bolivar  
**Actividad:** Prompt chaining y validación de interpretaciones con Ollama  
**Obra analizada:** *Cien años de soledad*, Gabriel García Márquez  
**Modelo disponible:** `qwen3.5:4b`  
**Fecha:** noviembre de 2026

## Objetivo

Este notebook demuestra un flujo reproducible para analizar un fragmento de *Cien años de soledad* con un modelo de lenguaje local. La temática es el **análisis literario asistido por LLM**: a partir de un pasaje del inicio de la novela, el sistema identifica elementos narrativos y después construye una interpretación respaldada por evidencias textuales.

La técnica central es **prompt chaining**: la salida estructurada de una primera etapa se convierte en la entrada controlada de una segunda etapa. Cada etapa tiene una responsabilidad limitada y la respuesta final se acepta únicamente si cumple validaciones de esquema, trazabilidad y correspondencia con el fragmento original.

> El notebook usa un fragmento breve almacenado en `nlp-transformers-capitulo-novela/data/capitulo_01.txt`. No se carga la novela completa. Si Ollama no está disponible o devuelve una salida inválida, el notebook detiene la ejecución explícitamente; así se evita presentar resultados escritos a mano como si fueran generados por el modelo.


## 1. Fundamentación y fuentes

La formulación separa la lectura analítica de la interpretación final. La primera etapa convierte el fragmento en observaciones verificables; la segunda propone una interpretación usando únicamente esas observaciones. Los ejemplos few-shot reducen la ambigüedad del formato esperado y la validación externa evita tratar cualquier texto generado como si fuera automáticamente una lectura correcta.

Fuentes consultadas:

1. García Márquez, Gabriel. *Cien años de soledad*. El fragmento se lee desde el archivo local `nlp-transformers-capitulo-novela/data/capitulo_01.txt`.
2. Wu et al. (2022), *AI Chains: Transparent and Controllable Human-AI Interaction by Chaining Large Language Model Prompts*: https://arxiv.org/abs/2202.01233
3. Brown et al. (2020), *Language Models are Few-Shot Learners*: https://arxiv.org/abs/2005.14165
4. Madaan et al. (2023), *Self-Refine: Iterative Refinement with Self-Feedback*: https://arxiv.org/abs/2303.17651
5. Ollama API documentation, generación de chat y formato JSON: https://github.com/ollama/ollama/blob/main/docs/api.md

Estas fuentes respaldan las decisiones observables en el código: dividir la tarea en etapas encadenadas, mostrar un ejemplo breve, solicitar una corrección iterativa cuando el contrato falla y comprobar la salida con reglas deterministas antes de presentarla como interpretación.


## 2. Preparación del entorno

Se utiliza únicamente la biblioteca estándar de Python para que el experimento sea portable. La función `ollama_chat` exige una respuesta real de Ollama y conserva visibles los errores de conexión o formato. No se utilizan respuestas hardcoded: todos los resultados mostrados en las etapas de análisis deben provenir del modelo local.


In [1]:
import json
import re
import urllib.error
import urllib.request
from datetime import datetime
from pprint import pprint

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3.5:4b"
TIMEOUT_SECONDS = 180
RUN_METADATA = {"model": MODEL, "ollama_available": False}


def http_json(path, payload=None, timeout=TIMEOUT_SECONDS):
    """Realiza una petición JSON sin depender de un SDK externo."""
    url = f"{OLLAMA_URL}{path}"
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=body,
        headers={"Content-Type": "application/json"} if body else {},
        method="POST" if body else "GET",
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        body = error.read().decode("utf-8", errors="replace")
        raise ValueError(f"Ollama HTTP {error.code}: {body[:200]}") from error


def available_models():
    """Consulta los modelos instalados y devuelve sus nombres."""
    try:
        tags = http_json("/api/tags", timeout=5)
        return [item.get("name", "") for item in tags.get("models", [])]
    except (OSError, ValueError, urllib.error.URLError):
        return []


models = available_models()
RUN_METADATA["ollama_available"] = MODEL in models
print("Fecha de ejecución:", datetime.now().isoformat(timespec="seconds"))
print("Modelos locales detectados:", models or "no disponibles")
print("Modelo seleccionado:", MODEL)
if not RUN_METADATA["ollama_available"]:
    raise RuntimeError(
        f"El modelo {MODEL!r} no está disponible en Ollama. "
        "Inicia Ollama y verifica `ollama list` antes de ejecutar el notebook."
    )


Fecha de ejecución: 2026-11-08T17:13:27
Modelos locales detectados: ['qwen3.5:4b']
Modelo seleccionado: qwen3.5:4b


In [2]:
def ollama_chat(messages, json_mode=False):
    """Solicita una respuesta real a Ollama y propaga los errores de generación."""
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "seed": 42},
    }
    if json_mode:
        payload["format"] = "json"
    response = http_json("/api/chat", payload)
    content = response.get("message", {}).get("content")
    if not isinstance(content, str) or not content.strip():
        raise ValueError("Ollama devolvió una respuesta vacía o sin contenido visible")
    return content.strip()


def parse_json_strict(raw_text):
    """Acepta JSON puro o JSON envuelto en fences Markdown, sin inventar contenido."""
    if not isinstance(raw_text, str):
        raise ValueError("La respuesta de Ollama no es texto")
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        value = json.loads(cleaned)
    except (json.JSONDecodeError, TypeError) as error:
        raise ValueError("La respuesta de Ollama no es JSON válido") from error
    if not isinstance(value, dict):
        raise ValueError("La respuesta JSON de Ollama no es un objeto")
    return value


## 3. Fragmento y contrato de datos

El fragmento corresponde al inicio de *Cien años de soledad* y se toma del archivo local `nlp-transformers-capitulo-novela/data/capitulo_01.txt`. Se trabaja únicamente con un pasaje breve para que el modelo pueda justificar cada observación. El contrato de la etapa 1 exige separar observaciones de inferencias: el modelo debe citar evidencias presentes y registrar qué información no aparece explícitamente.


In [3]:
from pathlib import Path

candidate_paths = [
    Path("../nlp-transformers-capitulo-novela/data/capitulo_01.txt"),
    Path("nlp-transformers-capitulo-novela/data/capitulo_01.txt"),
]
SOURCE_PATH = next((path for path in candidate_paths if path.exists()), None)
if SOURCE_PATH is None:
    raise FileNotFoundError("No se encontró capitulo_01.txt en las rutas esperadas")

source_text = SOURCE_PATH.read_text(encoding="utf-8")
start_marker = "Muchos años después, frente al pelotón de fusilamiento"
end_marker = "muchas cosas carecían de nombre, y"
start = source_text.find(start_marker)
end = source_text.find(end_marker, start)
if start < 0 or end < 0:
    raise ValueError("No se encontró el fragmento esperado en la fuente local")
fragment_end = end + len(end_marker)
fragment = source_text[start:fragment_end]

# La fuente es local y conocida; además se rechazan instrucciones incrustadas
# para que el texto analizado no pueda alterar el contrato de los prompts.
injection_pattern = re.compile(
    r"(ignora\s+(?:las\s+)?instrucciones|system\s*prompt|developer\s+message|"
    r"devuelve\s+solo\s+json|act[uú]a\s+como\s+un\s+asistente)",
    re.IGNORECASE,
)
if injection_pattern.search(fragment):
    raise ValueError("El fragmento contiene patrones sospechosos de prompt injection")

EXTRACTION_SCHEMA = {
    "required": {"setting", "characters", "conflict", "evidence", "missing_information"},
    "allowed": {"setting", "characters", "conflict", "evidence", "missing_information"},
}
EXTRACTION_METADATA = {
    "evidence_rule": "cada evidencia debe ser una cita exacta del fragmento",
    "source": str(SOURCE_PATH),
}

print("Obra: Cien años de soledad")
print("Fuente local leída:", SOURCE_PATH)
print("Caracteres leídos de la fuente:", len(source_text))
print("Caracteres del fragmento:", len(fragment))
print("\nFragmento seleccionado:\n", fragment)
print("\nContrato de extracción:")
pprint(EXTRACTION_SCHEMA)
print("Metadatos de procedencia:")
pprint(EXTRACTION_METADATA)


Obra: Cien años de soledad
Fuente local leída: ../nlp-transformers-capitulo-novela/data/capitulo_01.txt
Caracteres leídos de la fuente: 34837
Caracteres del fragmento: 452

Fragmento seleccionado:
 Muchos años después, frente al pelotón de fusilamiento, el coronel Aureliano Buendía había de recordar aquella tarde remota en que su padre lo llevó a conocer el hielo. Macondo era entonces una aldea de veinte casas de barro y cañabrava construidas a la orilla de un río de aguas diáfanas que se precipitaban por un lecho de piedras pulidas, blancas y enormes como huevos prehistóricos. El mundo era tan reciente, que muchas cosas carecían de nombre, y

Contrato de extracción:
{'allowed': {'characters',
             'conflict',
             'evidence',
             'missing_information',
             'setting'},
 'required': {'characters',
              'conflict',
              'evidence',
              'missing_information',
              'setting'}}
Metadatos de procedencia:
{'evidence_rule':

## 4. Cadena, etapa 1: lectura estructurada

La primera instrucción no pide una interpretación global. Su único objetivo es transformar el fragmento en observaciones JSON: ambiente, personajes, conflicto, citas de evidencia e información ausente. El ejemplo few-shot muestra cómo distinguir una observación textual de una inferencia no demostrada.


In [4]:
extraction = None
extraction_messages = [
    {
        "role": "system",
        "content": (
            "Eres un analista literario riguroso. Responde solo con JSON válido. "
            "Separa observaciones de interpretaciones y no inventes información. "
            "Cada elemento de evidence debe ser una cita exacta del fragmento. "
            "Si el conflicto no está explícito, describe la tensión observable sin declarar que no existe."
        ),
    },
    {
        "role": "user",
        "content": (
            "Ejemplo: 'El viajero cerró la puerta mientras la mujer esperaba en silencio.'\n"
            'Salida: {"setting":"interior de una casa","characters":["un viajero","una mujer"],'
            '"conflict":"separación no resuelta","evidence":["cerró la puerta",'
            '"esperaba en silencio"],"missing_information":["motivo de la separación"]}\n\n'
            f"Fragmento de Cien años de soledad: {fragment}\n"
            "Devuelve exactamente las claves setting, characters, conflict, evidence y missing_information."
        ),
    },
]

raw_extraction = ollama_chat(extraction_messages, json_mode=True)
extraction = parse_json_strict(raw_extraction)
print("Observaciones estructuradas de la etapa 1:")
pprint(extraction)


Observaciones estructuradas de la etapa 1:
{'characters': ['el coronel Aureliano Buendía'],
 'conflict': 'memoria del pasado frente a la realidad presente',
 'evidence': ['Muchos años después, frente al pelotón de fusilamiento',
              'recordar aquella tarde remota'],
 'missing_information': ['relación temporal entre el fusilamiento y el '
                         'recuerdo',
                         'motivo del fusilamiento'],
 'setting': 'frente al pelotón de fusilamiento'}


## 5. Validación de la etapa 1

La validación es independiente del modelo. Comprueba presencia de campos, tipos, valores permitidos y que no se hayan generado listas vacías en los campos que sustentan la decisión. Si falla, el pipeline se detiene con un error explícito; no se produce una respuesta final basada en datos no confiables.


In [5]:
def validate_extraction(value, fragment_text):
    """Valida claves, tipos internos y trazabilidad dentro del fragmento seleccionado."""
    if len(fragment_text) >= 2000:
        raise ValueError("validate_extraction espera un fragmento breve, no el capítulo completo")
    errors = []
    missing = EXTRACTION_SCHEMA["required"] - set(value)
    unexpected = set(value) - EXTRACTION_SCHEMA["allowed"]
    if missing:
        errors.append(f"faltan campos: {sorted(missing)}")
    if unexpected:
        errors.append(f"hay campos no autorizados: {sorted(unexpected)}")
    if not isinstance(value.get("setting"), str) or not value.get("setting", "").strip():
        errors.append("setting debe ser texto no vacío")
    for key in ("characters", "evidence", "missing_information"):
        if not isinstance(value.get(key), list):
            errors.append(f"{key} debe ser una lista")
        elif any(not isinstance(item, str) or not item.strip() for item in value[key]):
            errors.append(f"todos los elementos de {key} deben ser textos no vacíos")
    if not isinstance(value.get("characters"), list) or not value.get("characters"):
        errors.append("characters debe ser una lista no vacía")
    if not isinstance(value.get("conflict"), str) or not value.get("conflict", "").strip():
        errors.append("conflict debe ser texto no vacío")
    if not isinstance(value.get("evidence"), list) or not value.get("evidence"):
        errors.append("evidence debe ser una lista no vacía")
    if isinstance(value.get("evidence"), list):
        for quote in value["evidence"]:
            if quote not in fragment_text:
                errors.append(f"evidencia no encontrada literalmente: {quote!r}")
    return errors

extraction_errors = validate_extraction(extraction, fragment)
if extraction_errors:
    raise ValueError("Validación de extracción fallida: " + "; ".join(extraction_errors))
print("Validación de extracción: PASÓ")


Validación de extracción: PASÓ


## 6. Cadena, etapa 2: interpretación fundamentada

La segunda etapa recibe únicamente el JSON validado de la etapa 1, no el fragmento crudo. Esta restricción demuestra que la interpretación depende de observaciones previamente verificadas y evita que el modelo salte directamente de la fuente a una conclusión. Las citas de `evidence_used` permiten auditar qué observaciones sostienen la lectura.


In [6]:
final_response = None
response_messages = [
    {
        "role": "system",
        "content": (
            "Eres un crítico literario cuidadoso. Responde solo con JSON válido. "
            "Propón una interpretación razonable, pero no la presentes como una verdad única. "
            "Usa únicamente las observaciones validadas entregadas. "
            "Incluye interpretation, reasoning, evidence_used, alternative_reading y tone."
        ),
    },
    {
        "role": "user",
        "content": (
            "Obra: Cien años de soledad\n"
            "Observaciones validadas de la etapa 1:\n"
            + json.dumps(extraction, ensure_ascii=False, indent=2)
            + "\n\nEscribe una interpretación de 2 o 3 frases. En interpretation o reasoning incluye literalmente "
            "al menos una cita completa de evidence. En evidence_used copia exactamente fragmentos literales "
            "del campo evidence: sin comillas envolventes, sin puntos suspensivos y sin abreviarlos. "
            "Menciona una lectura alternativa prudente."
        ),
    },
]

raw_response = ollama_chat(response_messages, json_mode=True)
final_response = parse_json_strict(raw_response)
print("Interpretación de la etapa 2:")
pprint(final_response)


Interpretación de la etapa 2:
{'alternative_reading': 'Es posible interpretar el fusilamiento no como un '
                        'juicio final, sino como una liberación necesaria que '
                        'permite al coronel cerrar ciclos de violencia y '
                        'comenzar a procesar la soledad inherente a su linaje.',
 'evidence_used': ['Muchos años después, frente al pelotón de fusilamiento',
                   'recordar aquella tarde remota'],
 'interpretation': 'La narrativa sitúa al coronel Aureliano Buendía en un '
                   'momento de máxima tensión donde la memoria del pasado se '
                   'colisiona violentamente con la realidad presente, como lo '
                   "evidencia la frase 'Muchos años después, frente al pelotón "
                   "de fusilamiento'. Esta estructura temporal sugiere que el "
                   'fusilamiento no es solo un evento aislado, sino un '
                   'catalizador que reactiva ciclos histór

## 7. Validaciones finales y comparación conceptual

La última validación comprueba que la interpretación sea utilizable y esté anclada en citas del fragmento. También rechaza afirmaciones absolutas como “el autor demuestra definitivamente”, porque un fragmento breve admite más de una lectura razonable. La validación no decide si una interpretación es estéticamente buena; comprueba que sea trazable y que respete la evidencia disponible.


In [7]:
def validate_final_response(value, fragment_text, source_evidence=None):
    errors = []
    required = {"interpretation", "reasoning", "evidence_used", "alternative_reading", "tone"}
    missing = required - set(value)
    unexpected = set(value) - required
    if missing:
        errors.append(f"faltan campos: {sorted(missing)}")
    if unexpected:
        errors.append(f"hay campos no autorizados: {sorted(unexpected)}")
    for key in ("interpretation", "reasoning", "alternative_reading", "tone"):
        if not isinstance(value.get(key), str) or not value.get(key, "").strip():
            errors.append(f"{key} debe ser texto no vacío")
    if not isinstance(value.get("evidence_used"), list) or not value.get("evidence_used"):
        errors.append("evidence_used debe ser una lista no vacía")

    absolute_claim = re.compile(
        r"\b(demuestra(?:n)?(?: definitivamente)?|la única interpretación|"
        r"sin ninguna duda|indiscutiblemente|es incuestionable|no admite otra lectura|"
        r"es evidente que|claramente|es obvio|no hay duda)\b",
        re.IGNORECASE,
    )
    for key in ("interpretation", "reasoning", "alternative_reading"):
        if absolute_claim.search(value.get(key, "")):
            errors.append(f"{key} contiene una afirmación absoluta")

    evidence_set = set(source_evidence) if source_evidence is not None else None
    exact_quotes = []
    if isinstance(value.get("evidence_used"), list):
        for quote in value["evidence_used"]:
            if not isinstance(quote, str) or not quote.strip():
                errors.append("cada elemento de evidence_used debe ser texto no vacío")
                continue
            if evidence_set is not None and quote not in evidence_set:
                errors.append("evidence_used contiene observaciones que no estaban en la extracción")
            if quote not in fragment_text:
                errors.append(f"cita final no encontrada literalmente en el fragmento: {quote!r}")
            exact_quotes.append(quote)

    grounded_text = " ".join(value.get(key, "") for key in ("interpretation", "reasoning"))
    if exact_quotes and not any(quote in grounded_text for quote in exact_quotes):
        errors.append("interpretation o reasoning debe incluir literalmente al menos una evidencia seleccionada")
    return errors

semantic_fields = ("interpretation", "reasoning", "alternative_reading", "tone")
initial_response = json.loads(json.dumps(final_response))
final_response = initial_response
final_errors = validate_final_response(final_response, fragment, extraction["evidence"])
repair_attempts = 0

while final_errors and repair_attempts < 1:
    repair_attempts += 1
    print(f"La respuesta inicial no cumplió el contrato: {'; '.join(final_errors)}")
    affected_fields = set()
    for error in final_errors:
        for field in semantic_fields + ("evidence_used",):
            if field in error:
                affected_fields.add(field)
    if not affected_fields:
        affected_fields = set(semantic_fields + ("evidence_used",))
    repair_messages = [
        {
            "role": "system",
            "content": (
                "Corrige únicamente los campos afectados y devuelve JSON válido. "
                "Conserva sin cambios todos los demás campos. "
                "evidence_used debe usar copias exactas del campo evidence, sin comillas ni elipsis. "
                "Incluye literalmente al menos una evidencia completa en interpretation o reasoning. "
                "Las interpretaciones deben usar lenguaje prudente y apoyarse en la evidencia."
            ),
        },
        {
            "role": "user",
            "content": (
                "Campos autorizados para corregir: " + json.dumps(sorted(affected_fields)) + "\n\n"
                "Respuesta anterior:\n" + json.dumps(final_response, ensure_ascii=False, indent=2) + "\n\n"
                "Errores detectados:\n" + json.dumps(final_errors, ensure_ascii=False) + "\n\n"
                "Observaciones válidas:\n" + json.dumps(extraction, ensure_ascii=False, indent=2)
            ),
        },
    ]
    repaired_response = parse_json_strict(ollama_chat(repair_messages, json_mode=True))
    preserved_response = json.loads(json.dumps(final_response))
    for field in affected_fields:
        if field in repaired_response:
            preserved_response[field] = repaired_response[field]
    final_response = preserved_response
    final_errors = validate_final_response(final_response, fragment, extraction["evidence"])

RUN_METADATA.update({
    "repair_loop_activated": repair_attempts > 0,
    "repair_attempts": repair_attempts,
    "extraction_fields": sorted(extraction.keys()),
    "fragment_chars": len(fragment),
    "evidence_count": len(extraction.get("evidence", [])),
    "extraction_validation": "PASÓ",
    "final_validation": "PASÓ" if not final_errors else "FALLÓ",
})

if final_errors:
    raise ValueError(
        f"Validación final fallida después de {repair_attempts} intento(s) de reparación: "
        + "; ".join(final_errors)
    )

print("Validación final: PASÓ")
print("\nInterpretación lista para presentar:\n")
print(final_response["interpretation"])
print("\nJustificación:")
print(final_response["reasoning"])
print("\nLectura alternativa:")
print(final_response["alternative_reading"])
print("\nMetadatos de ejecución:")
pprint(RUN_METADATA)


Validación final: PASÓ

Interpretación lista para presentar:

La narrativa sitúa al coronel Aureliano Buendía en un momento de máxima tensión donde la memoria del pasado se colisiona violentamente con la realidad presente, como lo evidencia la frase 'Muchos años después, frente al pelotón de fusilamiento'. Esta estructura temporal sugiere que el fusilamiento no es solo un evento aislado, sino un catalizador que reactiva ciclos históricos y personales.

Justificación:
La obra utiliza la distancia temporal para contrastar la inevitabilidad del destino con la fugacidad de los recuerdos, donde 'recordar aquella tarde remota' actúa como el puente entre la experiencia vivida y su reinterpretación posterior. La ausencia explícita del motivo del fusilamiento invita a leerlo no como un acto político concreto, sino como una metáfora de la repetición histórica que define al pueblo Macondo.

Lectura alternativa:
Es posible interpretar el fusilamiento no como un juicio final, sino como una liberaci

## 8. Comparación experimental: one-shot frente a prompt chaining

Para evaluar si la cadena aporta algo, se ejecuta un baseline one-shot sobre el mismo fragmento y con el mismo modelo. El baseline solicita directamente la interpretación final y no tiene una etapa de extracción previa; por eso sus citas se comprueban contra el fragmento, mientras que la cadena también comprueba que provengan de una extracción validada. Se comparan ambos enfoques usando criterios explícitos de estructura, citas literales, fundamentación y afirmaciones absolutas.

La validación comprueba invenciones y formato, pero no garantiza que el modelo mencione todos los personajes o elementos presentes en el texto: una omisión puede pasar sin error. Esta limitación se registra para no confundir una salida formalmente válida con una extracción exhaustiva.


In [8]:
# Baseline: una sola instrucción solicita directamente la interpretación final.
one_shot_messages = [
    {
        "role": "system",
        "content": (
            "Eres un crítico literario cuidadoso. Responde solo con JSON válido. "
            "Analiza el fragmento directamente y no inventes citas. "
            "Incluye interpretation, reasoning, evidence_used, alternative_reading y tone."
        ),
    },
    {
        "role": "user",
        "content": (
            "Obra: Cien años de soledad\n"
            "Fragmento:\n" + fragment + "\n\n"
            "Devuelve una interpretación de 2 o 3 frases. En evidence_used incluye citas completas "
            "copiadas literalmente del fragmento y usa lenguaje prudente."
        ),
    },
]

raw_one_shot = ollama_chat(one_shot_messages, json_mode=True)
one_shot_response = parse_json_strict(raw_one_shot)
one_shot_evidence = (
    one_shot_response.get("evidence_used", [])
    if isinstance(one_shot_response.get("evidence_used"), list)
    else []
)
# El one-shot no tiene extracción previa: sus citas se comprueban contra el fragmento,
# pero no contra una lista de evidencias producida por otra etapa.
one_shot_errors = validate_final_response(
    one_shot_response,
    fragment,
    source_evidence=None,
)

comparison = {
    "one_shot_errors": len(one_shot_errors),
    "chained_errors": len(final_errors),
    "one_shot_evidence_count": len(one_shot_evidence),
    "chained_evidence_count": len(final_response.get("evidence_used", [])),
    "chained_repair_needed": RUN_METADATA["repair_loop_activated"],
    "chained_extraction_validated": RUN_METADATA["extraction_validation"] == "PASÓ",
    "chained_final_validated": RUN_METADATA["final_validation"] == "PASÓ",
}

print("Resultado one-shot:")
pprint(one_shot_response)
print("\nErrores one-shot:", one_shot_errors or "ninguno")
print("\nComparación experimental:")
pprint(comparison)
print(
    "\nLectura de la comparación: la cadena aporta trazabilidad adicional cuando "
    "conserva una extracción validada y la interpretación usa sus evidencias; "
    "la cantidad de errores por sí sola no prueba superioridad universal."
)


Resultado one-shot:
{'alternative_reading': 'La descripción del río y las piedras puede leerse '
                        'también como una metáfora de la fragilidad y la '
                        'eternidad simultánea de la existencia humana, donde '
                        'lo precario (barro) se encuentra con lo atemporal '
                        '(huevos prehistóricos).',
 'evidence_used': '"Muchos años después, frente al pelotón de fusilamiento, el '
                  'coronel Aureliano Buendía había de recordar aquella tarde '
                  'remota en que su padre lo llevó a conocer el hielo." y '
                  '"Macondo era entonces una aldea de veinte casas de barro y '
                  'cañabrava construidas a la orilla de un río de aguas '
                  'diáfanas que se precipitaban por un lecho de piedras '
                  'pulidas, blancas y enormes como huevos prehistóricos."',
 'interpretation': 'El fragmento establece un tono onírico y mítico que '
       

## 9. Interpretación de resultados y conclusiones

**Interpretación.** La primera etapa convierte el fragmento narrativo en observaciones verificables: ambiente, personajes, conflicto y citas exactas. La segunda etapa usa esas observaciones para construir una lectura interpretativa y declara una alternativa posible. El baseline one-shot permite observar qué ocurre cuando se solicita la interpretación directamente, sin esa separación intermedia.

**Conclusiones.** En este experimento, prompt chaining aporta una ruta auditable: cada paso tiene un contrato pequeño, la extracción se valida antes de interpretar y las evidencias utilizadas pueden rastrearse hasta el fragmento. La comparación con one-shot ayuda a identificar diferencias concretas en errores, trazabilidad y necesidad de reparación, pero una sola ejecución no demuestra una superioridad universal; para eso se necesitarían varios fragmentos, repeticiones y métricas agregadas.

**Limitaciones.** La validación detecta citas inventadas, formatos inválidos y afirmaciones no prudentes, pero no puede asegurar que el modelo haya enumerado todos los personajes o detalles relevantes. El análisis se realiza sobre un fragmento breve y no incorpora contexto histórico, biográfico o la obra completa. La temperatura 0 y la semilla fija favorecen la reproducibilidad, pero no eliminan la variabilidad entre modelos. Si Ollama no está disponible o el modelo devuelve un formato inválido, la ejecución se detiene en lugar de fabricar resultados.
